# LoRA Fine-Tuning SmolLM2-135M to Answer in JSON

**Goal:** teach the small base model SmolLM2-135M to answer a short question with **one compact JSON object, `{"answer":"..."}`, and then stop**. Training uses LoRA adapters on the attention projections only. On its own, the base model just continues the text: it writes prose, repeats itself and never produces JSON.

**Sections**
0. **Run configuration.** Data file, split, training on/off, adapter path, debugging on/off. This is the only cell to edit between runs.
1. **Environment.** PyTorch version and device (CPU only here).
2. **Base model.** Load SmolLM2-135M and its tokenizer.
3. **Data.** Split the data file into train/validation/test, inspect it, preview the prompt → target format and load it with 🤗 `datasets`.
4. **LoRA setup.** Wrap the model with PEFT and inspect what is trainable.
5. **Preprocessing.** Build prompt + JSON target, tokenize with the prompt masked out of the loss and EOS appended, then pad.
6. **Training.** Configure the 🤗 `Trainer`, train (if `RUN_TRAINING`) and save the adapter.
7. **Evaluation.** Reload the saved adapter, generate for held-out and brand-new questions, and compare with the base model on the same questions.
8. **Debugging (optional, `RUN_DEBUG`).** Token-boundary checks, first-token probabilities, adapter-weight checks, and loss and forced-prefix comparisons between the base and LoRA models.

---
## 0. Run configuration

Edit this cell, then **Run All**. Everything downstream reads from these values.

| Setting | Meaning |
|---|---|
| `DATA_FILE` | JSON list of records of the form `{"prompt": "<question>", "answer": "<plain-text answer>"}` |
| `TRAIN_FRAC` / `VAL_FRAC` / `TEST_FRAC` | Split fractions (must add up to 1.0). Train and validation are rounded, and test gets the rest. |
| `SEED` | Shuffle seed, so the split is reproducible |
| `DEDUPLICATE` | Drop exact-duplicate records before splitting, so a record can't land in both train and test |
| `SPLIT_PREFIX` / `SPLIT_FILES` | Names of the train / validation / test files written next to the notebook |
| `RUN_TRAINING` | `True`: train and save a new adapter to `ADAPTER_PATH`. `False`: skip training and saving, and evaluate the adapter already at `ADAPTER_PATH`. |
| `ADAPTER_PATH` | Where the adapter (and the per-epoch checkpoints) are saved, and where evaluation loads it from |
| `RUN_DEBUG` | `True`: run the diagnostic cells in section 8. `False`: skip them. |

Current settings: `data/lora_json_answer/smollm2_answer_250_fresh.json`, 250 records → 200 / 25 / 25.

In [83]:
# ===== Run configuration: edit here, then Run All =====

# DATA_FILE = "smollm2_json_lora_train_750.json"   # JSON list of {instruction, input, output}
DATA_FILE = "data/lora_json_answer/smollm2_answer_250_fresh.json"

# Split fractions (must sum to 1.0). Test gets whatever is left after rounding.
TRAIN_FRAC = 0.80
VAL_FRAC = 0.10
TEST_FRAC = 0.10

SEED = 42             # shuffle seed for a reproducible split
DEDUPLICATE = False   # True = drop exact-duplicate records before splitting

# Split files written next to the notebook
SPLIT_PREFIX = "smollm2_answer"
SPLIT_FILES = {
    name: f"{SPLIT_PREFIX}_{name}.json"
    for name in ["train", "validation", "test"]
}

# Training / adapter
RUN_TRAINING = True                        # False = skip trainer.train() and only evaluate an existing adapter
ADAPTER_PATH = "./smollm2-answer-lora-round3"  # final adapter is saved here after training, and loaded from here for evaluation

# Debugging
RUN_DEBUG = True    # True = run the diagnostic cells in section 8, False = skip them

assert abs(TRAIN_FRAC + VAL_FRAC + TEST_FRAC - 1.0) < 1e-9, "Split fractions must sum to 1.0"

print("Data file:  ", DATA_FILE)
print("Split:      ", f"{TRAIN_FRAC:.0%} train / {VAL_FRAC:.0%} val / {TEST_FRAC:.0%} test")
print("Seed:       ", SEED, "| Deduplicate:", DEDUPLICATE)
print("Split files:", list(SPLIT_FILES.values()))
print("Run training:", RUN_TRAINING, "| Adapter path:", ADAPTER_PATH)
print("Run debug:   ", RUN_DEBUG)

Data file:   data/lora_json_answer/smollm2_answer_250_fresh.json
Split:       80% train / 10% val / 10% test
Seed:        42 | Deduplicate: False
Split files: ['smollm2_answer_train.json', 'smollm2_answer_validation.json', 'smollm2_answer_test.json']
Run training: True | Adapter path: ./smollm2-answer-lora-round3


---
## 1. Environment check
Confirm the PyTorch version and whether a GPU is available. **CUDA is not available** (`2.14.1+cpu`), so everything runs on the CPU in fp32.

In [84]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Torch path:", torch.__file__)

PyTorch: 2.14.1+cpu
CUDA available: False
Torch path: /mnt/d/projects/kv-cache-experiment/.venv/lib/python3.12/site-packages/torch/__init__.py


Import the 🤗 Transformers classes used throughout (`AutoTokenizer`, `AutoModelForCausalLM`) and repeat the device check.

In [85]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.14.1+cpu
CUDA available: False


---
## 2. Load the base model

Load SmolLM2-135M (~134.5M parameters) and its tokenizer in `float32`. This is a **base** model, not an instruction-tuned one, so it continues text rather than following instructions. `model` is the object that LoRA wraps and trains in section 4, and `tokenizer` is used for all preprocessing.

In [86]:
model_name = "HuggingFaceTB/SmolLM2-135M"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
)

model.eval()

print("Model loaded:", model_name)
print("Parameters:", sum(p.numel() for p in model.parameters()))

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Model loaded: HuggingFaceTB/SmolLM2-135M
Parameters: 134515008


---
## 3. Data preparation

### 3.1 Split the dataset
Load `DATA_FILE`, optionally deduplicate it, shuffle it with `SEED` and cut it into train / validation / test by the configured fractions. Each split is written to its file in `SPLIT_FILES`, and `train_data` / `val_data` / `test_data` stay in memory. Section 7 uses `test_data` for held-out evaluation.

In [87]:
import json
import random

with open(DATA_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Loaded:", len(data), "records from", DATA_FILE)

if DEDUPLICATE:
    seen = set()
    unique = []
    for record in data:
        key = json.dumps(record, sort_keys=True, ensure_ascii=False)
        if key not in seen:
            seen.add(key)
            unique.append(record)
    print(f"Deduplicated: {len(data)} -> {len(unique)} records")
    data = unique

# Deterministic randomization
random.Random(SEED).shuffle(data)

n = len(data)
n_train = round(n * TRAIN_FRAC)
n_val = round(n * VAL_FRAC)

train_data = data[:n_train]
val_data = data[n_train:n_train + n_val]
test_data = data[n_train + n_val:]

assert train_data and val_data and test_data, "Every split needs at least one record"

for name, records in [
    ("train", train_data),
    ("validation", val_data),
    ("test", test_data),
]:
    with open(SPLIT_FILES[name], "w", encoding="utf-8") as f:
        json.dump(records, f, indent=2, ensure_ascii=False)

print("Train:", len(train_data))
print("Validation:", len(val_data))
print("Test:", len(test_data))

Loaded: 250 records from data/lora_json_answer/smollm2_answer_250_fresh.json
Train: 200
Validation: 25
Test: 25


### 3.2 Inspect the splits
Re-read each split file and print its first record. Each record is a short question (`prompt`) and a one-sentence plain-text `answer`. The data stores plain text, and the JSON wrapping is added during formatting.

In [88]:
import json

for name in ["train", "validation", "test"]:
    with open(SPLIT_FILES[name], "r", encoding="utf-8") as f:
        records = json.load(f)

    print(f"{name}: {len(records)} records")
    print("Example prompt:", records[0]["prompt"])
    print("Example answer:", records[0]["answer"])
    print()

train: 200 records
Example prompt: Can I import CSV files?
Example answer: Yes, CSV files can be imported

validation: 25 records
Example prompt: What is a batch size?
Example answer: Batch size is the number of examples processed together in one batch

test: 25 records
Example prompt: What is precision?
Example answer: Precision is the fraction of predicted positive results that are correct



### 3.3 Preview the prompt → target format
For one training example, show exactly what the model will see:

- **Prompt:** the question as-is, e.g. `Can I import CSV files?`
- **Target:** the answer wrapped as compact JSON, `{"answer":"Yes, CSV files can be imported"}` (`separators=(",", ":")`, so there are no spaces).

During training the target follows the prompt **directly, with no separator or "JSON:" cue**, so the model has to learn to emit `{"` immediately after the question mark. This cell only previews the format. The real formatting is done in section 5.

In [89]:
example = train_data[0]

prompt = example["prompt"].strip()

target = json.dumps(
    {"answer": example["answer"]},
    ensure_ascii=False,
    separators=(",", ":"),
)

print("PROMPT")
print("=" * 60)
print(prompt)

print("\nTARGET")
print("=" * 60)
print(target)

PROMPT
Can I import CSV files?

TARGET
{"answer":"Yes, CSV files can be imported"}


### 3.4 Load as a Hugging Face `DatasetDict`
Load the three split files (`data_files=SPLIT_FILES`) with `load_dataset("json", ...)` so they can be `.map()`-ed in section 5 and passed to the `Trainer`.

In [90]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files=SPLIT_FILES,
)

print(dataset)
print(dataset["train"][0])

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'answer'],
        num_rows: 200
    })
    validation: Dataset({
        features: ['prompt', 'answer'],
        num_rows: 25
    })
    test: Dataset({
        features: ['prompt', 'answer'],
        num_rows: 25
    })
})
{'prompt': 'Can I import CSV files?', 'answer': 'Yes, CSV files can be imported'}


### 3.5 Sanity check
Row counts, column names (`prompt`, `answer`) and the first record of each split.

In [91]:
for split in dataset:
    print(f"{split}: {len(dataset[split])} records")
    print("Columns:", dataset[split].column_names)
    print("First record:")
    print(dataset[split][0])
    print()

train: 200 records
Columns: ['prompt', 'answer']
First record:
{'prompt': 'Can I import CSV files?', 'answer': 'Yes, CSV files can be imported'}

validation: 25 records
Columns: ['prompt', 'answer']
First record:
{'prompt': 'What is a batch size?', 'answer': 'Batch size is the number of examples processed together in one batch'}

test: 25 records
Columns: ['prompt', 'answer']
First record:
{'prompt': 'What is precision?', 'answer': 'Precision is the fraction of predicted positive results that are correct'}



---
## 4. LoRA setup

### 4.1 Library versions
Check the PEFT and Accelerate versions.

In [92]:
import peft
import accelerate

print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)

PEFT: 0.21.1
Accelerate: 1.15.0


### 4.2 Find the target modules
List the attention projection layers (`q_proj`, `k_proj`, `v_proj`, `o_proj`) in all 30 decoder layers. These are the layers LoRA will adapt.

In [93]:
for name, module in model.named_modules():
    if any(x in name for x in ["q_proj", "k_proj", "v_proj", "o_proj"]):
        print(name, "->", module.__class__.__name__)

model.layers.0.self_attn.q_proj -> Linear
model.layers.0.self_attn.k_proj -> Linear
model.layers.0.self_attn.v_proj -> Linear
model.layers.0.self_attn.o_proj -> Linear
model.layers.1.self_attn.q_proj -> Linear
model.layers.1.self_attn.k_proj -> Linear
model.layers.1.self_attn.v_proj -> Linear
model.layers.1.self_attn.o_proj -> Linear
model.layers.2.self_attn.q_proj -> Linear
model.layers.2.self_attn.k_proj -> Linear
model.layers.2.self_attn.v_proj -> Linear
model.layers.2.self_attn.o_proj -> Linear
model.layers.3.self_attn.q_proj -> Linear
model.layers.3.self_attn.k_proj -> Linear
model.layers.3.self_attn.v_proj -> Linear
model.layers.3.self_attn.o_proj -> Linear
model.layers.4.self_attn.q_proj -> Linear
model.layers.4.self_attn.k_proj -> Linear
model.layers.4.self_attn.v_proj -> Linear
model.layers.4.self_attn.o_proj -> Linear
model.layers.5.self_attn.q_proj -> Linear
model.layers.5.self_attn.k_proj -> Linear
model.layers.5.self_attn.v_proj -> Linear
model.layers.5.self_attn.o_proj ->

### 4.3 Wrap the model with LoRA
`r = 8`, `lora_alpha = 16` (scale α/r = 2), dropout 0.05, applied to the four attention projections. The base weights are frozen, and only **921,600 parameters (0.68%)** are trainable.

`get_peft_model` modifies `model` **in place**: the original object now contains the LoRA layers too.

In [94]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

lora_model = get_peft_model(model, lora_config)

total_params = sum(p.numel() for p in lora_model.parameters())
trainable_params = sum(
    p.numel() for p in lora_model.parameters()
    if p.requires_grad
)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable %:           {100 * trainable_params / total_params:.4f}%")

Total parameters:     135,436,608
Trainable parameters: 921,600
Trainable %:           0.6805%


### 4.4 Inspect the PEFT wrapper
`PeftModelForCausalLM` wraps a `LoraModel`...

In [95]:
print(type(lora_model))
print("\nTop-level children:")
for name, module in lora_model.named_children():
    print(name, "->", type(module).__name__)

<class 'peft.peft_model.PeftModelForCausalLM'>

Top-level children:
base_model -> LoraModel


...which in turn wraps the original `LlamaForCausalLM`.

In [96]:
print(type(lora_model.base_model))
print("\nChildren of base_model:")

for name, module in lora_model.base_model.named_children():
    print(name, "->", type(module).__name__)

<class 'peft.tuners.lora.model.LoraModel'>

Children of base_model:
model -> LlamaForCausalLM


### 4.5 Look inside one LoRA layer
Layer 0's `q_proj` is now a `lora.Linear`: the frozen `base_layer` (576→576) plus a low-rank update `B·A`, where **A** is `8×576` and **B** is `576×8`. That is 9,216 parameters instead of 331,776 for a full update. B starts at **zero**, so before training the adapter changes nothing.

In [97]:
q_proj = lora_model.base_model.model.model.layers[0].self_attn.q_proj

print(q_proj)

print("\nLoRA A shape:", q_proj.lora_A["default"].weight.shape)
print("LoRA B shape:", q_proj.lora_B["default"].weight.shape)

print("\nLoRA A parameters:", q_proj.lora_A["default"].weight.numel())
print("LoRA B parameters:", q_proj.lora_B["default"].weight.numel())

lora.Linear(
  (base_layer): Linear(in_features=576, out_features=576, bias=False)
  (lora_dropout): ModuleDict(
    (default): Dropout(p=0.05, inplace=False)
  )
  (lora_A): ModuleDict(
    (default): Linear(in_features=576, out_features=8, bias=False)
  )
  (lora_B): ModuleDict(
    (default): Linear(in_features=8, out_features=576, bias=False)
  )
  (lora_embedding_A): ParameterDict()
  (lora_embedding_B): ParameterDict()
  (lora_magnitude_vector): ModuleDict()
)

LoRA A shape: torch.Size([8, 576])
LoRA B shape: torch.Size([576, 8])

LoRA A parameters: 4608
LoRA B parameters: 4608


### 4.6 Confirm what is frozen
The base `q_proj` weight is **not** trainable, and the LoRA A and B weights **are**.

In [98]:
print("Base Q projection trainable:",
      q_proj.base_layer.weight.requires_grad)

print("LoRA A trainable:",
      q_proj.lora_A["default"].weight.requires_grad)

print("LoRA B trainable:",
      q_proj.lora_B["default"].weight.requires_grad)

Base Q projection trainable: False
LoRA A trainable: True
LoRA B trainable: True


### 4.7 All trainable tensors
240 trainable tensors (30 layers × 4 projections × A and B). The `k_proj` / `v_proj` B matrices output 192 features instead of 576 because of grouped-query attention (3 KV heads). The total matches the 921,600 above.

In [99]:
trainable = [
    (name, param.numel(), tuple(param.shape))
    for name, param in lora_model.named_parameters()
    if param.requires_grad
]

for name, count, shape in trainable[:12]:
    print(f"{name:80s} {shape!s:18s} {count:,}")

print("\nTotal trainable parameters:", sum(x[1] for x in trainable))
print("Number of trainable tensors:", len(trainable))

base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight           (8, 576)           4,608
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight           (576, 8)           4,608
base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight           (8, 576)           4,608
base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight           (192, 8)           1,536
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight           (8, 576)           4,608
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight           (192, 8)           1,536
base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight           (8, 576)           4,608
base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight           (576, 8)           4,608
base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight           (8, 576)           4,608
base_model.model.model.layers.1.self_attn.q_pr

---
## 5. Preprocessing

### 5.1 Build prompt + target
Map every record to `prompt` (the stripped question), `target` (compact `{"answer": ...}` JSON) and `text` (the two concatenated, used only for printing).

In [100]:
def format_example(example):
    prompt = example["prompt"].strip()

    target = json.dumps(
        {"answer": example["answer"]},
        ensure_ascii=False,
        separators=(",", ":"),
    )

    return {
        "prompt": prompt,
        "target": target,
        "text": prompt + target,
    }


formatted_dataset = dataset.map(format_example)

print(formatted_dataset["train"][0]["text"])

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/25 [00:00<?, ? examples/s]

Map:   0%|          | 0/25 [00:00<?, ? examples/s]

Can I import CSV files?{"answer":"Yes, CSV files can be imported"}


### 5.2 Tokenize, masking the prompt out of the loss
Tokenize the prompt (with special tokens) and the target (without) **separately**, then concatenate them. Tokenizing separately puts the prompt/target boundary at an exact token position, and sections 8.1 and 8.2 verify this.

- **Labels** are `-100` for every prompt token, so the loss is computed **only on the JSON target**.
- An **EOS token is appended to the target** and kept in the labels, so the model learns to stop right after the closing `"}`.

In [101]:
def tokenize_example(example):
    prompt_tokens = tokenizer(
        example["prompt"],
        add_special_tokens=True,
    )

    target_tokens = tokenizer(
        example["target"],
        add_special_tokens=False,
    )

    # Explicitly teach the model where the answer ends.
    target_tokens["input_ids"].append(tokenizer.eos_token_id)

    input_ids = (
        prompt_tokens["input_ids"]
        + target_tokens["input_ids"]
    )

    labels = (
        [-100] * len(prompt_tokens["input_ids"])
        + target_tokens["input_ids"]
    )

    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": [1] * len(input_ids),
    }

tokenized_dataset = formatted_dataset.map(
    tokenize_example,
    remove_columns=formatted_dataset["train"].column_names,
)

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/25 [00:00<?, ? examples/s]

Map:   0%|          | 0/25 [00:00<?, ? examples/s]

### 5.3 Set a pad token
SmolLM2 has no pad token, so reuse EOS (`<|endoftext|>`, id 0) for padding.

In [102]:
print("Before:")
print("pad_token:", tokenizer.pad_token)
print("pad_token_id:", tokenizer.pad_token_id)
print("eos_token:", tokenizer.eos_token)
print("eos_token_id:", tokenizer.eos_token_id)

tokenizer.pad_token = tokenizer.eos_token

print("\nAfter:")
print("pad_token:", tokenizer.pad_token)
print("pad_token_id:", tokenizer.pad_token_id)

Before:
pad_token: None
pad_token_id: None
eos_token: <|endoftext|>
eos_token_id: 0

After:
pad_token: <|endoftext|>
pad_token_id: 0


### 5.4 Padding collator
A custom collator right-pads each batch to its longest sequence: `pad_token_id` for `input_ids`, `0` for `attention_mask` and `-100` for `labels`, so padding never contributes to the loss. The real EOS at the end of each target is still supervised, because it is part of the labels and not padding.

In [103]:
from dataclasses import dataclass
from typing import Any

import torch


@dataclass
class DataCollatorForCausalLM:
    tokenizer: Any
    label_pad_token_id: int = -100

    def __call__(self, features):
        max_length = max(len(x["input_ids"]) for x in features)

        input_ids = []
        attention_mask = []
        labels = []

        for x in features:
            padding = max_length - len(x["input_ids"])

            input_ids.append(
                x["input_ids"] + [self.tokenizer.pad_token_id] * padding
            )

            attention_mask.append(
                x["attention_mask"] + [0] * padding
            )

            labels.append(
                x["labels"] + [self.label_pad_token_id] * padding
            )

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


data_collator = DataCollatorForCausalLM(tokenizer)

print("Data collator ready")

Data collator ready


---
## 6. Training

### 6.1 Library versions
Transformers 5.17.0, Accelerate 1.15.0.

In [104]:
import accelerate
import transformers

print("Accelerate:", accelerate.__version__)
print("Accelerate path:", accelerate.__file__)
print("Transformers:", transformers.__version__)

Accelerate: 1.15.0
Accelerate path: /mnt/d/projects/kv-cache-experiment/.venv/lib/python3.12/site-packages/accelerate/__init__.py
Transformers: 5.17.0


### 6.2 Training configuration
- **10 epochs**, batch size 4 × gradient accumulation 4 = **effective batch 16**. With 200 training records that is 13 optimizer steps per epoch (**130 in total**).
- **Learning rate** 2e-4 with linear decay to 0 and no warmup. fp32, because this is CPU only.
- **Evaluate and save every epoch**, keeping the last 2 checkpoints. `output_dir = ADAPTER_PATH`, so the `checkpoint-*` folders land inside the adapter folder.

The `Trainer` is always built, which is cheap. Whether it actually trains is decided in the next cell.

In [105]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=ADAPTER_PATH,
    num_train_epochs=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    fp16=False,
    bf16=False,
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
)

print("Trainer ready")

Trainer ready


### 6.3 Train (only if `RUN_TRAINING`)
If `RUN_TRAINING = False`, this cell does nothing and evaluation uses the adapter already at `ADAPTER_PATH`.

**Latest run:** 130 steps in about 7 minutes on the CPU. Train loss went from 3.76 to 1.22, and validation loss from 3.36 to 1.43, flattening from about epoch 7. The loss covers the whole target, including the free-text answer, so it can't approach zero the way pure field extraction can. The structural tokens (`{"`, `answer`, `":"`, `"}`, EOS) are learned almost perfectly (see section 8).

In [106]:
if RUN_TRAINING:
    train_result = trainer.train()

    print("\nTraining complete.")
    print("Final training loss:", train_result.training_loss)
else:
    print("RUN_TRAINING = False: skipping training. Evaluation will use the existing adapter at", ADAPTER_PATH)

/mnt/d/projects/kv-cache-experiment/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,3.763903,3.358140
2,3.144510,2.563279
3,2.538076,1.939542
4,1.602344,1.614324
5,1.458701,1.522170
6,1.399538,1.483362
7,1.257716,1.459099
8,1.294762,1.443351
9,1.235501,1.433114
10,1.222568,1.431726


/mnt/d/projects/kv-cache-experiment/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/mnt/d/projects/kv-cache-experiment/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/mnt/d/projects/kv-cache-experiment/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/mnt/d/projects/kv-cache-experiment/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super()._


Training complete.
Final training loss: 1.8154839222247785


### 6.4 Save the adapter (only if `RUN_TRAINING`)
Write the final adapter (`adapter_model.safetensors` + `adapter_config.json`) to `ADAPTER_PATH`. Saving is skipped when training was skipped, so an untrained adapter never overwrites a good one.

In [107]:
# Only save after a real training run, so an untrained adapter never overwrites ADAPTER_PATH
if RUN_TRAINING:
    trainer.save_model(ADAPTER_PATH)
    print("Adapter saved to", ADAPTER_PATH)

Adapter saved to ./smollm2-answer-lora-round3


---
## 7. Evaluation

### 7.1 Reload the adapter
Load a fresh base model and attach the adapter from `ADAPTER_PATH`. `Trainable parameters: 0` is expected, because `PeftModel.from_pretrained` loads the adapter in inference mode.

⚠️ `PeftModel.from_pretrained` injects the LoRA layers **into `base_model_eval` in place**. After this cell, `base_model_eval` is **no longer a clean base model**, and calling it runs the adapter too (see 8.4). Any base comparison must use a freshly loaded model, as 7.4 and 8.6 do.

In [108]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

base_model_name = "HuggingFaceTB/SmolLM2-135M"
adapter_path = ADAPTER_PATH

tokenizer_eval = AutoTokenizer.from_pretrained(base_model_name)

base_model_eval = AutoModelForCausalLM.from_pretrained(
    base_model_name
)

lora_model_eval = PeftModel.from_pretrained(
    base_model_eval,
    adapter_path
)

lora_model_eval.eval()

print("Base model loaded")
print("LoRA adapter loaded")
print("Trainable parameters:", sum(
    p.numel() for p in lora_model_eval.parameters() if p.requires_grad
))

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Base model loaded
LoRA adapter loaded
Trainable parameters: 0


### 7.2 LoRA on held-out test questions
Defines `generate_lora` (greedy decoding that returns only the new tokens) and runs it on the first 3 `test_data` questions.

**Result:** every output is a single, valid `{"answer":"..."}` object that stops at EOS, so the format and stopping were learned. The answers are fluent and on topic but not always correct. For example, "precision" is defined wrongly compared with the dataset's answer. Factual accuracy is limited by the 135M base model, not by the adapter.

In [109]:
import torch


def generate_lora(model, tokenizer, prompt, max_new_tokens=128):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()


# Just inspect 3 examples visually
for i in range(3):
    example = test_data[i]
    prompt = example["prompt"].strip()

    output = generate_lora(
        lora_model_eval,
        tokenizer_eval,
        prompt,
    )

    print("=" * 80)
    print(f"CASE {i + 1}")
    print("-" * 80)
    print("PROMPT:")
    print(prompt)
    print()
    print("ADAPTER OUTPUT:")
    print(output)
    print()

CASE 1
--------------------------------------------------------------------------------
PROMPT:
What is precision?

ADAPTER OUTPUT:
{"answer":"Precision is the average of the individual precision of a set of predictions"}

CASE 2
--------------------------------------------------------------------------------
PROMPT:
What is a business key?

ADAPTER OUTPUT:
{"answer":"A business key is a unique identifier for a business"}

CASE 3
--------------------------------------------------------------------------------
PROMPT:
What is a reverse proxy?

ADAPTER OUTPUT:
{"answer":"A reverse proxy is a server that forwards traffic to a different server"}



### 7.3 LoRA on brand-new questions
Five questions that appear in neither the training nor the test data, to check that the format generalizes beyond memorized prompts. (`generate_lora` is redefined here with `max_new_tokens=64`.)

**Result:** all five again come out as one clean `{"answer":"..."}` object. Content quality varies: the load-balancer and index answers are good, and the authentication-vs-authorization answer is weak.

In [110]:
import torch

# New prompts not used in the training/test examples
new_prompts = [
    "What is the purpose of a load balancer?",
    "Why do databases use indexes?",
    "What is the difference between authentication and authorization?",
    "What is object storage?",
    "Why is data normalization used in relational databases?",
]


def generate_lora(model, tokenizer, prompt, max_new_tokens=64):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Remove the input prompt from the generated sequence
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()


# Run evaluation
for i, prompt in enumerate(new_prompts, start=1):

    output = generate_lora(
        lora_model_eval,
        tokenizer_eval,
        prompt,
    )

    print("=" * 80)
    print(f"CASE {i}")
    print("-" * 80)
    print("PROMPT:")
    print(prompt)
    print()
    print("ADAPTER OUTPUT:")
    print(output)
    print()

CASE 1
--------------------------------------------------------------------------------
PROMPT:
What is the purpose of a load balancer?

ADAPTER OUTPUT:
{"answer":"Load balancers distribute traffic across multiple servers"}

CASE 2
--------------------------------------------------------------------------------
PROMPT:
Why do databases use indexes?

ADAPTER OUTPUT:
{"answer":"Indexes allow data to be searched more quickly"}

CASE 3
--------------------------------------------------------------------------------
PROMPT:
What is the difference between authentication and authorization?

ADAPTER OUTPUT:
{"answer":"Authentication and authorization are two different levels of authentication"}

CASE 4
--------------------------------------------------------------------------------
PROMPT:
What is object storage?

ADAPTER OUTPUT:
{"answer":"Object storage stores data in objects"}

CASE 5
--------------------------------------------------------------------------------
PROMPT:
Why is data normal

### 7.4 Base model (no LoRA) on the same questions
Load a **completely fresh** base model, without the adapter, and run the same 8 questions (the 3 test questions + the 5 new ones).

**Result:** plain prose that repeats itself until `max_new_tokens`. There is no JSON and the model never stops. Compared with 7.2 and 7.3, this shows what the adapter added: the JSON wrapper and a clean stop.

In [111]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load a completely fresh base model — NO LoRA
base_model_name = "HuggingFaceTB/SmolLM2-135M"

tokenizer_base = AutoTokenizer.from_pretrained(base_model_name)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name
)

base_model.eval()


# The same 8 prompts used for the LoRA evaluation
prompts = [
    # Earlier 3
    "What is precision?",
    "What is a business key?",
    "What is a reverse proxy?",

    # New 5
    "What is the purpose of a load balancer?",
    "Why do databases use indexes?",
    "What is the difference between authentication and authorization?",
    "What is object storage?",
    "Why is data normalization used in relational databases?",
]


def generate_base(model, tokenizer, prompt, max_new_tokens=64):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Remove the original prompt
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()


# Generate using BASE MODEL ONLY
for i, prompt in enumerate(prompts, start=1):

    output = generate_base(
        base_model,
        tokenizer_base,
        prompt,
    )

    print("=" * 80)
    print(f"BASE MODEL - CASE {i}")
    print("-" * 80)
    print("PROMPT:")
    print(prompt)
    print()
    print("BASE MODEL OUTPUT:")
    print(output)
    print()

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

BASE MODEL - CASE 1
--------------------------------------------------------------------------------
PROMPT:
What is precision?

BASE MODEL OUTPUT:
Precision is the ability to achieve a specific result or result set.

Precision is a measure of how close a measurement is to the true value.

Precision is a measure of how close a measurement is to the true value.

Precision is a measure of how close a measurement

BASE MODEL - CASE 2
--------------------------------------------------------------------------------
PROMPT:
What is a business key?

BASE MODEL OUTPUT:
A business key is a set of instructions that a business uses to make sure that its customers are getting the goods and services that they need.

What is a business key?

A business key is a set of instructions that a business uses to make sure that its customers are getting the goods and services

BASE MODEL - CASE 3
--------------------------------------------------------------------------------
PROMPT:
What is a reverse proxy?

---
## 8. Debugging (optional)

Every cell in this section runs **only when `RUN_DEBUG = True`** in the run configuration. Otherwise each one prints a skip message.

These cells were used to work out whether the adapter is really doing anything. **Conclusion: it is.** Against a fresh base model, the probability of the first target token `{"` rises from about 1e-6 to 0.71–1.0 (8.6), and the loss on held-out targets drops by about 2.5–3 nats (8.8). The impression that "LoRA changes nothing" came from 8.4, which compared the LoRA model with `base_model_eval`. PEFT had already injected the adapter into that model, so both sides were the same model.

### 8.1 Token boundary for one example
Rebuild one training example by hand and print the tokens and labels around the prompt/target boundary. The 6 prompt tokens all have label `-100`. The 12 target tokens, including EOS, are supervised, and the first one is `{"` (id 39428).

In [112]:
if RUN_DEBUG:
    # Inspect exactly what the model is trained to predict
    # around the prompt -> target boundary.

    example = formatted_dataset["train"][0]

    prompt = example["prompt"]
    target = example["target"]

    prompt_tokens = tokenizer(
        prompt,
        add_special_tokens=True,
    )["input_ids"]

    target_tokens = tokenizer(
        target,
        add_special_tokens=False,
    )["input_ids"]

    target_tokens_with_eos = target_tokens + [tokenizer.eos_token_id]

    input_ids = prompt_tokens + target_tokens_with_eos
    labels = [-100] * len(prompt_tokens) + target_tokens_with_eos

    print("PROMPT")
    print("=" * 80)
    print(prompt)

    print("\nTARGET")
    print("=" * 80)
    print(target)

    print("\nTOKEN BOUNDARY")
    print("=" * 80)

    print("Prompt token count :", len(prompt_tokens))
    print("Target token count :", len(target_tokens_with_eos))

    print("\nLast prompt tokens:")
    for token_id in prompt_tokens[-10:]:
        print(
            token_id,
            repr(tokenizer.decode([token_id]))
        )

    print("\nFirst target tokens:")
    for token_id in target_tokens_with_eos[:15]:
        print(
            token_id,
            repr(tokenizer.decode([token_id]))
        )

    print("\nLABELS AROUND BOUNDARY")
    print("=" * 80)

    start = max(0, len(prompt_tokens) - 5)
    end = min(len(input_ids), len(prompt_tokens) + 15)

    for i in range(start, end):
        token = tokenizer.decode([input_ids[i]])
        label = labels[i]

        print(
            f"{i:4d} | "
            f"token={token!r:20s} | "
            f"label={label}"
        )
else:
    print("RUN_DEBUG = False: skipped")

PROMPT
Can I import CSV files?

TARGET
{"answer":"Yes, CSV files can be imported"}

TOKEN BOUNDARY
Prompt token count : 6
Target token count : 12

Last prompt tokens:
7306 'Can'
339 ' I'
752 ' import'
31632 ' CSV'
4577 ' files'
47 '?'

First target tokens:
39428 '{"'
11247 'answer'
25535 '":"'
10539 'Yes'
28 ','
31632 ' CSV'
4577 ' files'
416 ' can'
325 ' be'
13254 ' imported'
23597 '"}'
0 '<|endoftext|>'

LABELS AROUND BOUNDARY
   1 | token=' I'                 | label=-100
   2 | token=' import'            | label=-100
   3 | token=' CSV'               | label=-100
   4 | token=' files'             | label=-100
   5 | token='?'                  | label=-100
   6 | token='{"'                 | label=39428
   7 | token='answer'             | label=11247
   8 | token='":"'                | label=25535
   9 | token='Yes'                | label=10539
  10 | token=','                  | label=28
  11 | token=' CSV'               | label=31632
  12 | token=' files'             | label=4577


### 8.2 Boundary check across the whole training set
For every tokenized training example, confirm that the first supervised token is `{"`. No "BAD EXAMPLE" lines are printed, so no example has its prompt and target merged into a different token at the boundary.

In [113]:
if RUN_DEBUG:
    # Check the prompt -> target boundary across the entire training set.

    for i in range(len(tokenized_dataset["train"])):
        item = tokenized_dataset["train"][i]

        input_ids = item["input_ids"]
        labels = item["labels"]

        first_supervised = next(
            j for j, label in enumerate(labels)
            if label != -100
        )

        first_target_token = tokenizer.decode(
            [input_ids[first_supervised]]
        )

        if first_target_token != '{"':
            print(
                f"BAD EXAMPLE {i}: "
                f"first supervised token = {first_target_token!r}"
            )

    print("Boundary check complete.")
else:
    print("RUN_DEBUG = False: skipped")

Boundary check complete.


### 8.3 LoRA's first-token prediction on training examples
Run the full tokenized training sequence through `lora_model_eval` and show the top 5 predictions at the position just before the target. `{"` is ranked first with high probability (0.75, 1.00 and 1.00).

Note that this cell reassigns `model = lora_model_eval`. Reload the base model (section 2) before training again in the same session.

In [114]:
if RUN_DEBUG:
    import torch


    # Look at the model's prediction for the FIRST target token.
    # This is the critical transition:
    #
    #     "What is ...?"  ->  {"answer": ...


    model = lora_model_eval
    model.eval()

    for i in range(3):
        item = tokenized_dataset["train"][i]

        input_ids = torch.tensor(
            [item["input_ids"]],
            device=model.device,
        )

        attention_mask = torch.tensor(
            [item["attention_mask"]],
            device=model.device,
        )

        labels = torch.tensor(
            [item["labels"]],
            device=model.device,
        )

        with torch.no_grad():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )

        # The prediction at position N predicts token N+1.
        first_target_position = next(
            j for j, label in enumerate(item["labels"])
            if label != -100
        )

        logits = outputs.logits[0, first_target_position - 1]

        probs = torch.softmax(logits, dim=-1)

        top_probs, top_ids = torch.topk(probs, k=5)

        expected_id = item["input_ids"][first_target_position]

        print("=" * 80)
        print(f"TRAIN EXAMPLE {i}")
        print("PROMPT:", formatted_dataset["train"][i]["prompt"])
        print("TARGET:", formatted_dataset["train"][i]["target"])

        print("\nExpected first target token:")
        print(
            repr(tokenizer.decode([expected_id])),
            "| token id:", expected_id
        )

        print("\nTop 5 predictions:")
        for token_id, prob in zip(
            top_ids.tolist(),
            top_probs.tolist(),
        ):
            print(
                f"{prob:8.4f}  "
                f"{token_id:6d}  "
                f"{tokenizer.decode([token_id])!r}"
            )
else:
    print("RUN_DEBUG = False: skipped")

TRAIN EXAMPLE 0
PROMPT: Can I import CSV files?
TARGET: {"answer":"Yes, CSV files can be imported"}

Expected first target token:
'{"' | token id: 39428

Top 5 predictions:
  0.7500   39428  '{"'
  0.0156      76  '\\'
  0.0121   16005  '";'
  0.0121      18  '"'
  0.0107     476  ' "'
TRAIN EXAMPLE 1
PROMPT: What is Kubernetes?
TARGET: {"answer":"Kubernetes is a platform for orchestrating containerized workloads"}

Expected first target token:
'{"' | token id: 39428

Top 5 predictions:
  1.0000   39428  '{"'
  0.0006    9583  ' {"'
  0.0001     107  '{'
  0.0001   34026  "{'"
  0.0000    7353  " {'"
TRAIN EXAMPLE 2
PROMPT: What is object storage commonly used for?
TARGET: {"answer":"Object storage is commonly used for files, backups, logs, and data lake storage"}

Expected first target token:
'{"' | token id: 39428

Top 5 predictions:
  0.9961   39428  '{"'
  0.0015    9583  ' {"'
  0.0002   28292  '={"'
  0.0002   34026  "{'"
  0.0001   26754  '{\\'


### 8.4 Base vs. LoRA first-token probability (misleading)
This compares `base_model_eval` with `lora_model_eval` and reports a **change of exactly 0**. That isn't evidence that the adapter is inactive. `base_model_eval` is the object that `PeftModel.from_pretrained` injected the adapter into (7.1), so both calls run the LoRA weights. Cell 8.6 repeats the comparison correctly.

(The `input_ids` computed at the top of the loop are unused, because the cell re-tokenizes the prompt.)

In [115]:
if RUN_DEBUG:
    import torch


    base_model = base_model_eval
    adapter_model = lora_model_eval

    base_model.eval()
    adapter_model.eval()

    for i in range(3):
        item = tokenized_dataset["train"][i]

        input_ids = torch.tensor(
            [item["input_ids"][:len(item["input_ids"]) - len(item["labels"]) + 
             next(j for j, x in enumerate(item["labels"]) if x != -100)]],
            device=adapter_model.device,
        )

        # Simpler: use the original prompt directly.
        prompt = formatted_dataset["train"][i]["prompt"]

        inputs = tokenizer_eval(
            prompt,
            return_tensors="pt",
        ).to(adapter_model.device)

        expected_token_id = 39428  # tokenizer token for '{"'

        with torch.no_grad():
            base_logits = base_model(**inputs).logits[0, -1]
            adapter_logits = adapter_model(**inputs).logits[0, -1]

        base_prob = torch.softmax(base_logits, dim=-1)[expected_token_id].item()
        adapter_prob = torch.softmax(adapter_logits, dim=-1)[expected_token_id].item()

        print("=" * 80)
        print(f"EXAMPLE {i}")
        print("PROMPT:", prompt)
        print(f'Expected first token: {tokenizer_eval.decode([expected_token_id])!r}')
        print()
        print(f"Base model probability  : {base_prob:.6f}")
        print(f"LoRA model probability  : {adapter_prob:.6f}")
        print(f"Change                  : {adapter_prob - base_prob:+.6f}")
else:
    print("RUN_DEBUG = False: skipped")

EXAMPLE 0
PROMPT: Can I import CSV files?
Expected first token: '{"'

Base model probability  : 0.710938
LoRA model probability  : 0.710938
Change                  : +0.000000
EXAMPLE 1
PROMPT: What is Kubernetes?
Expected first token: '{"'

Base model probability  : 1.000000
LoRA model probability  : 1.000000
Change                  : +0.000000
EXAMPLE 2
PROMPT: What is object storage commonly used for?
Expected first token: '{"'

Base model probability  : 0.996094
LoRA model probability  : 0.996094
Change                  : +0.000000


### 8.5 Check the adapter is loaded and non-zero
List the active adapters (`default`) and every LoRA tensor's mean and max absolute value. All LoRA parameters are non-zero. In particular the **B** matrices, which start at zero, now have mean |w| of about 3–5e-3, so training did move them.

In [116]:
if RUN_DEBUG:
    # Verify that the LoRA adapter is actually active
    # and contains non-zero learned weights.

    print("Active adapters:", lora_model_eval.active_adapters)

    print("\nLoRA parameters:")
    total = 0
    nonzero = 0

    for name, param in lora_model_eval.named_parameters():
        if "lora_" in name:
            total += param.numel()
            nonzero += (param.detach().abs() > 0).sum().item()

            print(
                name,
                "shape=", tuple(param.shape),
                "mean_abs=", param.detach().abs().mean().item(),
                "max_abs=", param.detach().abs().max().item(),
            )

    print("\nTotal LoRA parameters:", total)
    print("Non-zero LoRA parameters:", nonzero)
    print(
        "Non-zero ratio:",
        f"{nonzero / total:.4%}" if total else "N/A"
    )
else:
    print("RUN_DEBUG = False: skipped")

Active adapters: ['default']

LoRA parameters:
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight shape= (8, 576) mean_abs= 0.021580960601568222 max_abs= 0.05130566656589508
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight shape= (576, 8) mean_abs= 0.003429076401516795 max_abs= 0.011450063437223434
base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight shape= (8, 576) mean_abs= 0.021615615114569664 max_abs= 0.05180167779326439
base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight shape= (192, 8) mean_abs= 0.003928787540644407 max_abs= 0.012469562701880932
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight shape= (8, 576) mean_abs= 0.02178763411939144 max_abs= 0.05326776206493378
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight shape= (192, 8) mean_abs= 0.0045997039414942265 max_abs= 0.011650838889181614
base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weig

### 8.6 Base vs. LoRA first-token probability, done correctly
Repeat 8.4 against a **freshly loaded** base model. The probability of `{"` straight after the question:

| Example | Base | LoRA |
|---|---|---|
| Can I import CSV files? | 0.000002 | 0.71 |
| What is Kubernetes? | 0.000001 | 1.00 |
| What is object storage commonly used for? | ~0 | 1.00 |

The adapter is clearly active and is responsible for starting the JSON.

In [117]:
if RUN_DEBUG:
    # Load a completely fresh copy of the base model for a clean comparison.
    # Do NOT reuse base_model_eval because it was wrapped by PeftModel.

    from transformers import AutoModelForCausalLM
    import torch

    fresh_base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name
    )

    fresh_base_model.eval()
    lora_model_eval.eval()

    for i in range(3):
        prompt = formatted_dataset["train"][i]["prompt"]

        inputs = tokenizer_eval(
            prompt,
            return_tensors="pt",
        )

        with torch.no_grad():
            base_logits = fresh_base_model(**inputs).logits[0, -1]
            lora_logits = lora_model_eval(**inputs).logits[0, -1]

        base_probs = torch.softmax(base_logits, dim=-1)
        lora_probs = torch.softmax(lora_logits, dim=-1)

        expected_token_id = tokenizer_eval.convert_tokens_to_ids('{"')

        base_prob = base_probs[expected_token_id].item()
        lora_prob = lora_probs[expected_token_id].item()

        print("=" * 80)
        print(f"EXAMPLE {i}")
        print("PROMPT:", prompt)
        print()
        print("Expected first target token:", repr('{"'))
        print()
        print(f"Base model probability : {base_prob:.6f}")
        print(f"LoRA model probability : {lora_prob:.6f}")
        print(f"Change                 : {lora_prob - base_prob:+.6f}")
else:
    print("RUN_DEBUG = False: skipped")

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

EXAMPLE 0
PROMPT: Can I import CSV files?

Expected first target token: '{"'

Base model probability : 0.000002
LoRA model probability : 0.710938
Change                 : +0.710936
EXAMPLE 1
PROMPT: What is Kubernetes?

Expected first target token: '{"'

Base model probability : 0.000001
LoRA model probability : 1.000000
Change                 : +0.999999
EXAMPLE 2
PROMPT: What is object storage commonly used for?

Expected first target token: '{"'

Base model probability : 0.000000
LoRA model probability : 0.996094
Change                 : +0.996094


### 8.7 What competes with `{"` for the first token
The top 10 next-token candidates after the question, under LoRA. `{"` dominates, and the runners-up are other JSON-like starts (` {"`, `{'`, `={"`, `"`), so whatever the model picks, it is trying to open a JSON object.

In [118]:
if RUN_DEBUG:
    # Inspect the complete competition for the first target token.

    for i in range(3):
        prompt = formatted_dataset["train"][i]["prompt"]

        inputs = tokenizer_eval(
            prompt,
            return_tensors="pt",
        )

        with torch.no_grad():
            logits = lora_model_eval(**inputs).logits[0, -1]

        probs = torch.softmax(logits, dim=-1)

        top_probs, top_ids = torch.topk(probs, k=10)

        print("=" * 80)
        print(f"EXAMPLE {i}")
        print("PROMPT:", prompt)
        print()

        for rank, (token_id, prob) in enumerate(
            zip(top_ids.tolist(), top_probs.tolist()),
            start=1,
        ):
            print(
                f"{rank:2d}. "
                f"{prob:.6f}  "
                f"{tokenizer_eval.decode([token_id])!r}"
            )
else:
    print("RUN_DEBUG = False: skipped")

EXAMPLE 0
PROMPT: Can I import CSV files?

 1. 0.710938  '{"'
 2. 0.016724  '\\'
 3. 0.014771  '";'
 4. 0.013000  '"'
 5. 0.011475  ' Answer'
 6. 0.011475  ' "'
 7. 0.010132  '['
 8. 0.008423  ' ['
 9. 0.007874  ' ('
10. 0.007416  ' \\'
EXAMPLE 1
PROMPT: What is Kubernetes?

 1. 1.000000  '{"'
 2. 0.000488  ' {"'
 3. 0.000066  '{'
 4. 0.000048  "{'"
 5. 0.000043  ' {'
 6. 0.000033  '={"'
 7. 0.000028  " {'"
 8. 0.000024  ' Answer'
 9. 0.000016  '#'
10. 0.000014  ' {{'
EXAMPLE 2
PROMPT: What is object storage commonly used for?

 1. 0.996094  '{"'
 2. 0.001320  ' {"'
 3. 0.000230  '={"'
 4. 0.000179  "{'"
 5. 0.000139  '{\\'
 6. 0.000139  'Answer'
 7. 0.000123  " {'"
 8. 0.000109  ' {'
 9. 0.000096  '{'
10. 0.000084  ' Answer'


### 8.8 Loss on held-out targets: fresh base vs. LoRA
Teacher-forced loss on the full target (`{"answer":...}` + EOS) for 3 **test** questions. LoRA lowers the loss by about 2.5–3 nats per token, e.g. 4.15 → 1.15 on "What is precision?". The adapter generalizes beyond the training set, not just on the first token.

In [119]:
if RUN_DEBUG:
    import torch


    def target_loss(model, example):
        prompt = example["prompt"].strip()

        target = json.dumps(
            {"answer": example["answer"]},
            ensure_ascii=False,
            separators=(",", ":"),
        )

        prompt_tokens = tokenizer_eval(
            prompt,
            add_special_tokens=True,
        )["input_ids"]

        target_tokens = tokenizer_eval(
            target,
            add_special_tokens=False,
        )["input_ids"]

        target_tokens.append(tokenizer_eval.eos_token_id)

        input_ids = prompt_tokens + target_tokens

        labels = (
            [-100] * len(prompt_tokens)
            + target_tokens
        )

        input_ids = torch.tensor(
            [input_ids],
            device=model.device,
        )

        labels = torch.tensor(
            [labels],
            device=model.device,
        )

        with torch.no_grad():
            outputs = model(
                input_ids=input_ids,
                labels=labels,
            )

        return outputs.loss.item()


    for i in range(3):
        example = test_data[i]

        base_loss = target_loss(
            fresh_base_model,
            example,
        )

        lora_loss = target_loss(
            lora_model_eval,
            example,
        )

        print("=" * 80)
        print(f"EXAMPLE {i}")
        print("PROMPT:", example["prompt"])
        print("TARGET:", json.dumps(
            {"answer": example["answer"]},
            ensure_ascii=False,
            separators=(",", ":"),
        ))
        print()
        print(f"Base loss : {base_loss:.4f}")
        print(f"LoRA loss : {lora_loss:.4f}")
        print(f"Change    : {lora_loss - base_loss:+.4f}")
else:
    print("RUN_DEBUG = False: skipped")

EXAMPLE 0
PROMPT: What is precision?
TARGET: {"answer":"Precision is the fraction of predicted positive results that are correct"}

Base loss : 4.1517
LoRA loss : 1.1516
Change    : -3.0001
EXAMPLE 1
PROMPT: What is a business key?
TARGET: {"answer":"A business key identifies a business entity using meaningful source-system values"}

Base loss : 5.2914
LoRA loss : 2.8182
Change    : -2.4732
EXAMPLE 2
PROMPT: What is a reverse proxy?
TARGET: {"answer":"A reverse proxy receives client requests and forwards them to backend services"}

Base loss : 3.6595
LoRA loss : 1.1914
Change    : -2.4681


### 8.9 Forced `{"` prefix: LoRA continuation
Append `{"` to the prompt and let the LoRA model continue. It completes `answer":"…"}` and emits `<|endoftext|>`, producing the same outputs as normal generation in 7.2. This confirms that the first token was never the bottleneck.

In [120]:
if RUN_DEBUG:
    import torch


    for i in range(3):
        prompt = test_data[i]["prompt"].strip()

        inputs = tokenizer_eval(
            prompt,
            return_tensors="pt",
        )

        # Force the first generated token to be '{"'
        forced_prefix = tokenizer_eval(
            '{"',
            add_special_tokens=False,
        )["input_ids"]

        forced_prefix = torch.tensor(
            [forced_prefix],
            device=lora_model_eval.device,
        )

        input_ids = torch.cat(
            [
                inputs["input_ids"],
                forced_prefix,
            ],
            dim=1,
        )

        attention_mask = torch.ones_like(input_ids)

        with torch.no_grad():
            outputs = lora_model_eval.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=40,
                do_sample=False,
                pad_token_id=tokenizer_eval.eos_token_id,
                eos_token_id=tokenizer_eval.eos_token_id,
            )

        generated = outputs[0][inputs["input_ids"].shape[1]:]

        print("=" * 80)
        print(f"CASE {i + 1}")
        print("PROMPT:", prompt)
        print()
        print("FORCED '{'+'\"' + MODEL CONTINUATION:")
        print(tokenizer_eval.decode(
            generated,
            skip_special_tokens=False,
        ))
else:
    print("RUN_DEBUG = False: skipped")

CASE 1
PROMPT: What is precision?

FORCED '{'+'"' + MODEL CONTINUATION:
{"answer":"Precision is the average of the individual precision of a set of predictions"}<|endoftext|>
CASE 2
PROMPT: What is a business key?

FORCED '{'+'"' + MODEL CONTINUATION:
{"answer":"A business key is a unique identifier for a business"}<|endoftext|>
CASE 3
PROMPT: What is a reverse proxy?

FORCED '{'+'"' + MODEL CONTINUATION:
{"answer":"A reverse proxy is a server that forwards traffic to a different server"}<|endoftext|>


### 8.10 Forced `{"` prefix: base model continuation
The same forced start with the fresh base model. It produces junk (`{" />`, `{"/tbody/tr/…`, URL maps), never writes an `answer` key and never stops. Even when given the opening token, the base model can't do the task. Both the format and the stopping come from the adapter.

In [121]:
if RUN_DEBUG:
    import torch


    for i in range(3):
        prompt = test_data[i]["prompt"].strip()

        inputs = tokenizer_eval(
            prompt,
            return_tensors="pt",
        )

        # Force the first generated token to '{"'
        forced_prefix = tokenizer_eval(
            '{"',
            add_special_tokens=False,
        )["input_ids"]

        forced_prefix = torch.tensor(
            [forced_prefix],
            device=fresh_base_model.device,
        )

        input_ids = torch.cat(
            [
                inputs["input_ids"],
                forced_prefix,
            ],
            dim=1,
        )

        attention_mask = torch.ones_like(input_ids)

        with torch.no_grad():
            outputs = fresh_base_model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=40,
                do_sample=False,
                pad_token_id=tokenizer_eval.eos_token_id,
                eos_token_id=tokenizer_eval.eos_token_id,
            )

        generated = outputs[0][inputs["input_ids"].shape[1]:]

        print("=" * 80)
        print(f"BASE CASE {i + 1}")
        print("PROMPT:", prompt)
        print()
        print('FORCED \'{"\' + BASE MODEL CONTINUATION:')
        print(tokenizer_eval.decode(
            generated,
            skip_special_tokens=False,
        ))
else:
    print("RUN_DEBUG = False: skipped")

BASE CASE 1
PROMPT: What is precision?

FORCED '{"' + BASE MODEL CONTINUATION:
{" />

Precision is the ability to make a measurement with a high degree of accuracy. It is the ability to make a measurement with a high degree of accuracy. It is the ability to make
BASE CASE 2
PROMPT: What is a business key?

FORCED '{"' + BASE MODEL CONTINUATION:
{"/tbody/tr/tr/tr/tr/tr/tr/tr/tr/tr/tr/tr/tr/tr/tr/tr/tr/tr/tr/
BASE CASE 3
PROMPT: What is a reverse proxy?

FORCED '{"' + BASE MODEL CONTINUATION:
{"/": "http://www.google.com/","/": "http://www.yahoo.com/","/": "http://www.yahoo.com/","/":


### 8.11 Training log
Print the loss logged every 10 steps from `trainer.state.log_history`. This reads the trainer from the current session, so it prints only the header when `RUN_TRAINING = False`.

In [122]:
if RUN_DEBUG:
    print("Training log:")
    print("=" * 80)

    for entry in trainer.state.log_history:
        if "loss" in entry:
            print(
                f"step={entry.get('step', '?'):>4}  "
                f"loss={entry['loss']:.4f}"
            )
else:
    print("RUN_DEBUG = False: skipped")

Training log:
step=  10  loss=3.7639
step=  20  loss=3.1445
step=  30  loss=2.5381
step=  40  loss=2.0251
step=  50  loss=1.6023
step=  60  loss=1.4587
step=  70  loss=1.3995
step=  80  loss=1.3852
step=  90  loss=1.2577
step= 100  loss=1.2948
step= 110  loss=1.2355
step= 120  loss=1.2734
step= 130  loss=1.2226
